# Регрессия CC50 с подбором гиперпараметров

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Загрузка данных
df = pd.read_excel('../data/Данные_для_курсовои_Классическое_МО.xlsx')
df = df.drop(columns=['Unnamed: 0'])
df = df.dropna()

In [3]:
# Подготовка данных
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = np.log1p(df['CC50, mM'])                          # логарифмируем из-за скошенности

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# Удаляем коррелированные признаки с порогом 0.95 (только на train)
corr = X_train.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(abs(upper[col]) > 0.95)]

# Применяем удаление и к train, и к test
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print(f"Удалено признаков: {len(to_drop)}")

Удалено признаков: 33


Применим следующие модели:  
- "LinearRegression"
- "RandomForest"
- "GradientBoosting"
- "CatBoost"
- "XGBoost"

**Подбор гиперпараметров не проводил из-за высокой вычислительной стоимости: полный перебор занимал более 3 часов.**

In [5]:
# ВСЕ 5 МОДЕЛЕЙ
models = {
    "LinearRegression": Pipeline([
        ('scaler', RobustScaler()),       # Логистическая регрессия чувствительна к масштабу
        ('model', LinearRegression())
    ]),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_seed=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)
}

# Обучение и сравнение
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_log = model.predict(X_test)
    
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)
    
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)

print("\n" + "="*48)
print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ ВСЕХ 5 МОДЕЛЕЙ:")
print("="*48)
print(results_df.to_string(index=False))


РЕЗУЛЬТАТЫ СРАВНЕНИЯ ВСЕХ 5 МОДЕЛЕЙ:
           model        MAE       RMSE        R2
        CatBoost 314.357314 553.504016  0.349647
         XGBoost 307.194272 562.361712  0.328665
    RandomForest 318.506746 577.180435  0.292819
GradientBoosting 336.808933 588.057776  0.265913
LinearRegression 422.470305 759.950019 -0.225964


**Итог** 

Лучшая модель — XGBoost  

MAE = 307.19 → самая маленькая ошибка  
RMSE = 562.36 → тоже лучший результат  
R² = 0.329 → самый высокий  

Это означает:  
модель лучше всех предсказывает и объясняет данные.

**Вывод**

По результатам сравнительного анализа моделей наилучшее качество продемонстрировала модель XGBoost, обеспечившая минимальные значения метрик MAE и RMSE, а также наибольшее значение коэффициента детерминации (R² = 0.329). Это указывает на то, что данная модель наиболее точно предсказывает целевую переменную и лучше других описывает зависимости в данных.

Модель CatBoost показала сопоставимые результаты и заняла второе место, в то время как Random Forest и Gradient Boosting продемонстрировали более низкое качество. Линейная регрессия оказалась наименее эффективной, что подтверждается отрицательным значением R², свидетельствующим о неспособности модели адекватно описывать зависимость между признаками и целевой переменной.

В целом полученные результаты можно оценить как умеренные: модель XGBoost объясняет около 32.9% вариации целевой переменной, что указывает на наличие потенциала для дальнейшего повышения качества за счёт оптимизации гиперпараметров, улучшения признакового пространства и расширения обучающей выборки.